In [67]:
### TXREG Daten ziehen   ins df Dataframe
 
import requests
import os
import pandas as pd
from io import StringIO
import json

# Define the URL and credentials
url = "https://cloud.h-da.de/public.php/webdav"
username = "DNQpTLHqBQzbkFL"

with open('secret.json', 'r') as f:
    data = json.load(f)
    password = data['IDEN_DATA_SHARE_PW']

# Make the GET request with basic authentication
response = requests.get(url, auth=(username, password), headers={"X-Requested-With": "XMLHttpRequest"})

# Check if the request was successful
if response.status_code == 200:
    # Read the CSV data
    df = pd.read_csv(StringIO(response.text))
    print('Data retrieved successfully')
else:
    print(f"Failed to retrieve data: {response.status_code}")

Data retrieved successfully


### Daten analyse

In [68]:
# Gruppiere die Spalten nach ihren Datentypen
datatypes = df.dtypes

# Iteriere über jeden einzigartigen Datentyp
for dtype in datatypes.unique():
    print(f"Spalten mit Datentyp {dtype}  ({len(datatypes[datatypes == dtype].index.tolist())} Merkmale):")
    print(datatypes[datatypes == dtype].index.tolist())
    print()  # Leerzeile zur besseren Übersicht

print('Shape: ')
print(df.shape)
n_1 = df.shape[0]

Spalten mit Datentyp float64  (9 Merkmale):
['time', 'donor_age_years', 'donor_height_cm', 'donor_weight_kg', 'donor_creatinin_umol_per_l', 'recipient_age_years', 'recipient_height_cm', 'recipient_weight_kg', 'recipient_dialysis_years']

Spalten mit Datentyp int64  (3 Merkmale):
['event', 'recipient_pra', 'transplant_cold_ischemia_time_min']

Spalten mit Datentyp object  (5 Merkmale):
['donor_sex', 'donor_death_reason', 'recipient_sex', 'recipient_diagnosis', 'destination']

Spalten mit Datentyp bool  (7 Merkmale):
['donor_diabetes', 'donor_hypertension', 'donor_smoking', 'donor_hcv', 'recipient_bloodtransfusion', 'recipient_hcv', 'transplant_abo_compat']

Shape: 
(17016, 24)


In [69]:
# Zeilen entfernen mit time <= 0

df = df[df['time']>0]
print(f'{n_1-df.shape[0]} Zeilen entfernt')
print('\nShape nach time>0: ')
print(df.shape)

1230 Zeilen entfernt

Shape nach time>0: 
(15786, 24)


In [70]:
### Kategorische Merkmale mit zu vielen Ausprägungen entfernen/umwandeln

# transplant_abo_compat hat 2 ausprägungen, wovon eine nur 3 mal vorkommt --- > daher merkmal entfernen
# recipient_diagnosis   248 ausprägungen  -- merkmal rausnehmen, tipp von prof. dr. jahn
# donor_death_reason    41  ausprägungen  -- merkmal rausnehmen, tipp von prof. dr. jahn
df = df.drop(['recipient_diagnosis', 'donor_death_reason','transplant_abo_compat'], axis=1)

# recipient_pra 0 oder 1 setzen und in bool umwandeln, 0 falls 0 sonst 1, tipp von prof. dr. jahn
df['recipient_pra'] = df['recipient_pra'].apply(lambda x: x != 0)

In [71]:
# Merkmal destination, ausprägungen local dünn besetzt  --> zeilen mit der ausprägung rausnehmen
event_counts = df.groupby('destination')['event'].value_counts().unstack(fill_value=0)
#print(event_counts)
df = df[df['destination'] != 'Local']

In [72]:
df.head(1)

,time,event,donor_age_years,donor_sex,donor_height_cm,donor_weight_kg,donor_creatinin_umol_per_l,donor_diabetes,donor_hypertension,donor_smoking,...,recipient_age_years,recipient_sex,recipient_height_cm,recipient_weight_kg,recipient_bloodtransfusion,recipient_dialysis_years,recipient_hcv,recipient_pra,transplant_cold_ischemia_time_min,destination
0,1826.25,0,70.0,male,175.0,80.0,75.1,False,False,False,...,65.034908,female,166.5,71.5,False,8.873374,False,0,329,Regional


In [73]:
# recipient_pra         101 ausprägungen   aber laut pdf nur 2 möglich    -- binarisieren mit  falls 0 dann true sonst false


df.head(1)

,time,event,donor_age_years,donor_sex,donor_height_cm,donor_weight_kg,donor_creatinin_umol_per_l,donor_diabetes,donor_hypertension,donor_smoking,...,recipient_age_years,recipient_sex,recipient_height_cm,recipient_weight_kg,recipient_bloodtransfusion,recipient_dialysis_years,recipient_hcv,recipient_pra,transplant_cold_ischemia_time_min,destination
0,1826.25,0,70.0,male,175.0,80.0,75.1,False,False,False,...,65.034908,female,166.5,71.5,False,8.873374,False,False,329,Regional


In [74]:
# Merkmale zum trainieren des Classifiers festlegen
X_columns = df.columns.tolist()
X_columns.remove('time')
X_columns.remove('event')

### Train Test Split

In [75]:
# teile die daten in trainings- und testdaten auf nach cut bei 3 jahren
tau = 3* 365
test_size = 0.3
seed = 42

from utils import create_train_test_data
df_train, df_test,n_events_after_cut_train,portion_censored_after_cut_train,n_events_after_cut_test , portion_censored_after_cut_test = create_train_test_data(tau=tau, data = df, X_columns=X_columns,  seed=seed, test_size=test_size)

print('Train:')
print(f'Anteil der Zensierten beobachtungen nach dem cut bei tau: {round(portion_censored_after_cut_train*100,2)}%')
print(f'Anteil der Events nach dem cut bei tau: {round(n_events_after_cut_train/df_train.shape[0]*100,2)}%')

print('\nTest:')
print(f'Anteil der Zensierten beobachtungen nach dem cut bei tau: {round(portion_censored_after_cut_test*100,2)}%')
print(f'Anteil der Events nach dem cut bei tau: {round(n_events_after_cut_test/df_test.shape[0]*100,2)}%')

Train:
Anteil der Zensierten beobachtungen nach dem cut bei tau: 19.42%
Anteil der Events nach dem cut bei tau: 19.04%

Test:
Anteil der Zensierten beobachtungen nach dem cut bei tau: 20.1%
Anteil der Events nach dem cut bei tau: 17.88%


### Bagged decision Tree Classifier

In [76]:
from utils import DecisionTreeBaggingClassifier
from utils import ipc_weighted_mse

B = 1000
max_depth = 4
min_samples_split = 5
max_features = 'log2'      # auto, sqrt, None


params_rf = {   'n_estimators':B,                        
                'max_depth':max_depth,
                'min_samples_split':min_samples_split,
                'max_features': max_features,
                'random_state':  seed,
                'weighted_bootstrapping': True, }

# Dummy-Variablen für Training und Test-Daten erstellen
df_train_dummy = pd.get_dummies(df_train, drop_first=True)
df_test_dummy = pd.get_dummies(df_test, drop_first=True)

# Sicherstellen, dass beide Datasets die gleichen Spalten haben
df_test_dummy = df_test_dummy.reindex(columns=df_train_dummy.columns)

clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train_dummy.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train_dummy["survived"].values,
    sample_weights=df_train_dummy["weights_ipcw"].values,
)

_ , pred  =clf.predict_proba(df_test_dummy.drop(
            ["weights_ipcw", "survived", "time", "event"], axis=1
        ).values)

# Evaluation auf Testdaten
rf_mse_ipcw = ipc_weighted_mse(
    y_true=df_test_dummy["survived"].values, 
    y_pred=pred,
    sample_weight=df_test_dummy["weights_ipcw"].values,
)

print(f"IPCW MSE: {rf_mse_ipcw}")



# pdf noch mitreinnehmen , mse_ipcw das gleiche wie der ipc brier scrore, lukas naych referenz fragen
# lukas fragen wefgen  Datenquelle  txREg dataset



IPCW MSE: 0.14212228754266962


In [77]:
0.14211367298948233

0.14211367298948233